# Factor Pricing: High-Dimensional Factor Extraction and Dynamic Asset Pricing

This notebook walks through the complete research pipeline interactively.

**Contents**
1. Setup and data loading
2. Exploratory data analysis
3. Latent factor extraction (PCA + Bai-Ng criterion)
4. Observed factor model (Fama-French 5)
5. Forecasting experiments (OLS / Ridge / Lasso)
6. Dynamic rolling forecaster
7. Model evaluation and comparison
8. Portfolio construction and backtesting

In [ ]:
import sys, os
# Navigate to project root so imports work
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    sys.path.insert(0, os.path.join(project_root, 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

import config
print('Config loaded. Date range:', config.START_DATE, '-', config.END_DATE)
print('N_FACTORS:', config.N_FACTORS, '| ROLLING_WINDOW:', config.ROLLING_WINDOW)

## 1. Data Loading

In [ ]:
from src.data_pipeline import (
    download_equity_data, download_etf_data,
    download_ff_factors, download_fred_data,
    build_panel, save_panel, load_panel,
)

RESULTS_DIR = os.path.join(project_root, 'results')
DATA_DIR    = os.path.join(project_root, 'data')
PANEL_PATH  = os.path.join(DATA_DIR, config.PANEL_FILE)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR,    exist_ok=True)

FORCE_DOWNLOAD = False   # set True to re-download

if not FORCE_DOWNLOAD and os.path.exists(PANEL_PATH):
    panel = load_panel(PANEL_PATH)
    print('Loaded cached panel:', panel.shape)
else:
    equity_prices = download_equity_data(config.EQUITY_TICKERS, config.START_DATE, config.END_DATE)
    etf_prices    = download_etf_data(config.SECTOR_ETFS,        config.START_DATE, config.END_DATE)
    ff_factors    = download_ff_factors(config.START_DATE, config.END_DATE)
    macro_df      = download_fred_data(config.START_DATE, config.END_DATE)

    panel = build_panel(equity_prices, etf_prices, ff_factors, macro_df)
    save_panel(panel, PANEL_PATH)
    print('Panel built and saved:', panel.shape)

panel.head(3)

## 2. Exploratory Data Analysis

In [ ]:
from src.eda import (
    plot_correlation_heatmap, plot_eigenvalue_scree,
    plot_cumulative_variance, summary_stats, print_factor_loadings,
)

ret_cols = panel.columns[panel.columns.str.endswith('_ret')]
returns  = panel[ret_cols]

stats = summary_stats(returns)
print('Summary statistics (first 10 assets):')
stats.head(10)

In [ ]:
plot_correlation_heatmap(returns, results_dir=RESULTS_DIR)
print('Correlation heatmap saved.')

In [ ]:
plot_eigenvalue_scree(returns, n_components=20, results_dir=RESULTS_DIR)
plot_cumulative_variance(returns, n_components=20, results_dir=RESULTS_DIR)
print('Scree and variance plots saved.')

## 3. Latent Factor Extraction (PCA + Bai-Ng)

In [ ]:
from src.factor_models import LatentFactorModel, bai_ng_criterion, ObservedFactorModel

bn = bai_ng_criterion(returns, max_factors=config.MAX_FACTORS, results_dir=RESULTS_DIR)
print('Bai-Ng optimal k:', bn)

In [ ]:
lfm = LatentFactorModel(n_factors=config.N_FACTORS)
lfm.fit(returns)

latent_factors = lfm.transform(returns)
print('Latent factor returns shape:', latent_factors.shape)
print('Explained variance per factor:', np.round(lfm.explained_variance_ratio_ * 100, 2))

loadings = print_factor_loadings(lfm.pca, lfm.feature_names, n_factors=config.N_FACTORS)

In [ ]:
latent_factors.plot(subplots=True, figsize=(14, 10), title='Latent Factor Returns')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'latent_factor_returns.png'), dpi=150)
plt.show()

## 4. Observed Factor Model (FF5 + Macro)

In [ ]:
ff5_cols   = [c for c in ['Mkt-RF','SMB','HML','RMW','CMA','RF'] if c in panel.columns]
macro_cols = [c for c in config.FRED_SERIES.keys() if c in panel.columns]

observed_factors = panel[ff5_cols + macro_cols].copy()

ofm = ObservedFactorModel(add_constant=True)
ofm.fit(returns, observed_factors)

betas = ofm.get_factor_exposures()
print('Beta matrix shape:', betas.shape)
betas.T.head()

In [ ]:
# Heatmap of factor betas
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(betas.T, cmap='RdBu_r', center=0, linewidths=0.3, ax=ax)
ax.set_title('Observed Factor Betas (assets × factors)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'observed_factor_betas.png'), dpi=150)
plt.show()

## 5. Forecasting Experiments

In [ ]:
from src.regression_models import run_forecasting_experiment, build_feature_matrix

macro_df = panel[macro_cols] if macro_cols else pd.DataFrame(index=panel.index)

forecast_results = run_forecasting_experiment(
    panel_df         = panel,
    latent_factors   = latent_factors,
    observed_factors = observed_factors,
    macro_df         = macro_df,
    train_ratio      = config.TRAIN_RATIO,
    val_ratio        = config.VAL_RATIO,
    lags             = config.FEATURE_LAGS,
)
print('Forecasting results computed for models:', list(forecast_results.keys()))

In [ ]:
from src.evaluation import compare_models_table

for split in ('train', 'val', 'test'):
    compare_models_table(forecast_results, split=split)

## 6. Dynamic Rolling Forecaster

In [ ]:
from src.dynamic_models import RollingFactorForecaster, KalmanFactorModel

rff = RollingFactorForecaster(
    window     = config.ROLLING_WINDOW,
    n_factors  = config.N_FACTORS,
    model_type = 'ridge',
    lags       = [1, 5],
)
rolling_preds, rolling_actuals = rff.fit_predict(returns, macro_df)
print('Rolling predictions:', len(rolling_preds))

In [ ]:
from src.evaluation import plot_predictions_vs_actuals, compute_metrics

m = compute_metrics(rolling_actuals.values, rolling_preds.values)
print('Rolling forecaster metrics:', {k: round(v, 4) for k, v in m.items()})

plot_predictions_vs_actuals(
    rolling_actuals, rolling_preds,
    title='Rolling Ridge: Predictions vs Actuals',
    results_dir=RESULTS_DIR,
)
print('Plot saved.')

In [ ]:
# Kalman filter smoothing of latent factors
kf = KalmanFactorModel(n_iter=10)
kf.fit(latent_factors)
smoothed = kf.smooth()

fig, axes = plt.subplots(config.N_FACTORS, 1, figsize=(14, 3 * config.N_FACTORS))
for i, col in enumerate(latent_factors.columns):
    axes[i].plot(latent_factors.index, latent_factors[col], alpha=0.5, label='Raw', color='steelblue')
    axes[i].plot(smoothed.index, smoothed[col], label='Smoothed', color='darkorange', linewidth=1.5)
    axes[i].set_title(f'{col}: Raw vs Kalman-Smoothed')
    axes[i].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'kalman_smoothed_factors.png'), dpi=150)
plt.show()

## 7. Model Evaluation

In [ ]:
from src.evaluation import diebold_mariano_test, plot_rolling_r2

# Diebold-Mariano test: rolling model vs naive
e_model = (rolling_actuals - rolling_preds).values
e_naive = rolling_actuals.values
dm = diebold_mariano_test(e_model, e_naive)
print('Diebold-Mariano test (rolling vs naive 0-forecast):')
print(f"  stat = {dm['dm_stat']:.4f},  p-value = {dm['p_value']:.4f}")

In [ ]:
plot_rolling_r2(
    {'Rolling Ridge': rolling_preds},
    rolling_actuals,
    window=63,
    results_dir=RESULTS_DIR,
)
print('Rolling R² plot saved.')

## 8. Portfolio Construction and Backtesting

In [ ]:
from src.portfolio import (
    mean_variance_portfolio, factor_mimicking_portfolio,
    backtest_portfolio, compare_portfolios,
)

# Factor-mimicking portfolios
latent_w_mat = factor_mimicking_portfolio(returns, latent_factors)
obs_w_mat    = factor_mimicking_portfolio(returns, observed_factors[ff5_cols])

latent_weights = latent_w_mat.iloc[:, 0]
obs_weights    = obs_w_mat.iloc[:, 0]  if not obs_w_mat.empty else pd.Series()

ew_weights = pd.Series(
    np.ones(len(returns.columns)) / len(returns.columns),
    index=returns.columns
)

if obs_weights.empty or obs_weights.isna().all():
    obs_weights = ew_weights.copy()

summary = compare_portfolios(
    latent_factor_weights   = latent_weights,
    observed_factor_weights = obs_weights,
    equal_weights           = ew_weights,
    returns_df              = returns,
    results_dir             = RESULTS_DIR,
)
summary

In [ ]:
# Mean-variance portfolio
sample_ret = returns.iloc[-config.ROLLING_WINDOW:]
exp_ret    = sample_ret.mean()
cov_mat    = sample_ret.cov()
mv_w       = mean_variance_portfolio(exp_ret, cov_mat, risk_aversion=config.RISK_AVERSION)

mv_result  = backtest_portfolio(mv_w, returns)
print('Mean-Variance Portfolio:')
for k, v in mv_result.items():
    if not isinstance(v, pd.Series):
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
print('All outputs saved to:', RESULTS_DIR)
print('Files:', sorted(os.listdir(RESULTS_DIR)))